In [1]:
# ===== P5 CELL 1: Connect Google Drive =====
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ===== P5 CELL 2: Settings, data loaders and a resumable "run and save" helper =====
import os, json, time, hashlib, datetime, resource, warnings
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore", category=UserWarning)

BASE = os.environ.get("BETH_BASE", "/content/drive/MyDrive/BETH_Triage")
FEAT_DIR = os.path.join(BASE, "02_data_processed", "features")
MODEL_DIR = os.path.join(BASE, "03_models")
SCORE_DIR = os.path.join(BASE, "04_scores")
LOG_DIR = os.path.join(BASE, "07_logs")
for d in (MODEL_DIR, SCORE_DIR):
    os.makedirs(d, exist_ok=True)

with open(os.path.join(BASE, "config.json")) as fh:
    CONFIG = json.load(fh)
proto = CONFIG["protocol"]
with open(os.path.join(BASE, proto["file"]), "rb") as fh:
    assert hashlib.sha256(fh.read()).hexdigest() == proto["sha256"], "Protocol file changed - STOP"
PLAN = proto["plan"]
MODELS = PLAN["models"]
SEEDS = PLAN["model_seeds"]["repeats"]
PRIMARY_SEED = PLAN["model_seeds"]["primary"]
FEATURE_SETS = ["F_main", "F_paper", "F_no_user", "F_raw"]
print("Protocol", proto["version"], "verified unchanged")

# Only identifier columns are kept. Labels (sus, evil) are deliberately NOT loaded in this phase.
ID_COLS = ["userId", "processName", "eventName"]
ids = {s: pd.read_pickle(os.path.join(FEAT_DIR, f"meta_{s}.pkl"))[ID_COLS] for s in ["train", "val", "test"]}
es_idx = np.load(os.path.join(FEAT_DIR, "ae_early_stopping_idx.npy"))

def load_X(fs, split):
    return np.load(os.path.join(FEAT_DIR, f"{fs}_{split}.npy"))

def peak_ram_gb():
    return round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6, 2)

def run_config(method, fs, seed, fit_and_score):
    """Fits one configuration, saves its validation and test scores. Skips it if already saved."""
    name = f"{method}__{fs}__s{seed}"
    path = os.path.join(SCORE_DIR, name + ".npz")
    if os.path.exists(path):
        print(f"  {name}: already done - skipped")
        return
    t0 = time.time()
    model, score_fn = fit_and_score(fs, seed)
    fit_s = time.time() - t0
    s_val = score_fn("val")
    t1 = time.time()
    s_test = score_fn("test")
    score_s = time.time() - t1
    assert np.isfinite(s_val).all() and np.isfinite(s_test).all(), f"non-finite scores in {name}"
    np.savez_compressed(path, val=s_val.astype(np.float64), test=s_test.astype(np.float64))
    if model is not None:
        joblib.dump(model, os.path.join(MODEL_DIR, name + ".joblib"))
    info = {"config": name, "method": method, "feature_set": fs, "seed": seed,
            "fit_seconds": round(fit_s, 1),
            "score_seconds_per_10k": round(score_s / len(ids["test"]) * 10000, 3),
            "peak_ram_gb_so_far": peak_ram_gb(),
            "finished_at": datetime.datetime.now().isoformat(timespec="seconds")}
    with open(os.path.join(LOG_DIR, "P5_run_log.jsonl"), "a") as fh:
        fh.write(json.dumps(info) + "\n")
    print(f"  {name}: fit {fit_s:,.1f}s, scoring {info['score_seconds_per_10k']}s per 10k rows")

def configs_for(method):
    """Primary feature set with all 5 seeds; ablation feature sets with the primary seed (decision D4)."""
    return [("F_main", s) for s in SEEDS] + [(fs, PRIMARY_SEED) for fs in FEATURE_SETS[1:]]

print("Configurations per model:", configs_for("x"))

Protocol v1 verified unchanged
Configurations per model: [('F_main', 42), ('F_main', 43), ('F_main', 44), ('F_main', 45), ('F_main', 46), ('F_paper', 42), ('F_no_user', 42), ('F_raw', 42)]


In [3]:
# ===== P5 CELL 3: Record implementation decision D4 (before any model is trained) =====
with open(os.path.join(BASE, "config.json")) as fh:
    config = json.load(fh)
if not any(d.get("id") == "D4" for d in config["decisions"]):
    config["decisions"].append({
        "id": "D4",
        "decision": "Implementation details for Phase 5, fixed before training.",
        "details": [
            "Seed repeats (42-46) are run on the primary feature set F_main; ablation feature sets use seed 42.",
            "Baselines random (seed 42), userId rule and rarity do not depend on the feature set.",
            "Rarity: V = number of distinct (processName, eventName) pairs in training.",
            "Scores are oriented so that higher = more anomalous: IF uses -score_samples, "
            "OCSVM uses -decision_function, autoencoder uses per-record mean squared error.",
            "OCSVM and autoencoder inputs are standardised with a StandardScaler fitted on training rows only.",
            "OCSVM seed controls its 50,000-row training subsample.",
            "Autoencoder implemented with scikit-learn MLPRegressor (same layers, ReLU, Adam, lr 0.001, "
            "batch 1024), trained one epoch at a time on training rows outside the saved 10% early-stopping "
            "rows; training stops after 3 epochs without improvement (max 30) and the best epoch is kept.",
            "Labels are not loaded during Phase 5.",
        ],
        "recorded_at": datetime.datetime.now().isoformat(timespec="seconds"),
    })
    with open(os.path.join(BASE, "config.json"), "w") as fh:
        json.dump(config, fh, indent=2)
    print("Decision D4 recorded")
else:
    print("Decision D4 already recorded")

Decision D4 recorded


In [4]:
# ===== P5 CELL 4: Baselines - random, userId rule, rarity =====
def fit_random(fs, seed):
    def score(split):
        return np.random.default_rng([seed, 0 if split == "val" else 1]).random(len(ids[split]))
    return None, score

def fit_rule(fs, seed):
    def score(split):
        return (ids[split]["userId"].values >= 1000).astype(np.float64)
    return None, score

def fit_rarity(fs, seed):
    tr = ids["train"]
    pair_counts = (tr["processName"] + "|" + tr["eventName"]).value_counts()
    N, V = len(tr), len(pair_counts)
    def score(split):
        pairs = ids[split]["processName"] + "|" + ids[split]["eventName"]
        c = pairs.map(pair_counts).fillna(0).values
        return -np.log((c + 1) / (N + V))
    return {"pair_counts": pair_counts.to_dict(), "N": N, "V": V}, score

print("Baselines:")
run_config("random", "none", MODELS["random"]["seed"], fit_random)
run_config("rule_userId", "none", 0, fit_rule)
run_config("rarity", "none", 0, fit_rarity)

Baselines:
  random__none__s42: fit 0.0s, scoring 0.0s per 10k rows
  rule_userId__none__s0: fit 0.0s, scoring 0.0s per 10k rows
  rarity__none__s0: fit 0.2s, scoring 0.002s per 10k rows


In [5]:
# ===== P5 CELL 5: Isolation Forest (5 seeds on F_main + 3 ablation feature sets) =====
from sklearn.ensemble import IsolationForest
p = MODELS["isolation_forest"]

def fit_iforest(fs, seed):
    m = IsolationForest(n_estimators=p["n_estimators"], max_samples=p["max_samples"],
                        max_features=p["max_features"], random_state=seed, n_jobs=-1)
    m.fit(load_X(fs, "train"))
    return m, lambda split: -m.score_samples(load_X(fs, split))

print("Isolation Forest:")
for fs, seed in configs_for("isolation_forest"):
    run_config("iforest", fs, seed, fit_iforest)

Isolation Forest:
  iforest__F_main__s42: fit 5.7s, scoring 0.29s per 10k rows
  iforest__F_main__s43: fit 3.4s, scoring 0.107s per 10k rows
  iforest__F_main__s44: fit 2.7s, scoring 0.107s per 10k rows
  iforest__F_main__s45: fit 2.1s, scoring 0.105s per 10k rows
  iforest__F_main__s46: fit 2.4s, scoring 0.111s per 10k rows
  iforest__F_paper__s42: fit 2.2s, scoring 0.062s per 10k rows
  iforest__F_no_user__s42: fit 5.2s, scoring 0.09s per 10k rows
  iforest__F_raw__s42: fit 2.5s, scoring 0.063s per 10k rows


In [6]:
# ===== P5 CELL 6: One-Class SVM (5 seeds on F_main + 3 ablation feature sets) =====
from sklearn.svm import OneClassSVM
p = MODELS["ocsvm"]

def fit_ocsvm(fs, seed):
    Xtr = load_X(fs, "train")
    scaler = StandardScaler().fit(Xtr)
    rng = np.random.default_rng(seed)
    sub = rng.choice(len(Xtr), size=p["train_subsample"], replace=False)
    m = OneClassSVM(kernel=p["kernel"], gamma=p["gamma"], nu=p["nu"], cache_size=1000)
    m.fit(scaler.transform(Xtr[sub]))
    def score(split):
        X = scaler.transform(load_X(fs, split))
        return -np.concatenate([m.decision_function(X[i:i + 20000]) for i in range(0, len(X), 20000)])
    return {"scaler": scaler, "model": m}, score

print("One-Class SVM (this is the slowest model):")
for fs, seed in configs_for("ocsvm"):
    run_config("ocsvm", fs, seed, fit_ocsvm)

One-Class SVM (this is the slowest model):
  ocsvm__F_main__s42: fit 13.5s, scoring 0.79s per 10k rows
  ocsvm__F_main__s43: fit 6.5s, scoring 0.844s per 10k rows
  ocsvm__F_main__s44: fit 7.3s, scoring 0.872s per 10k rows
  ocsvm__F_main__s45: fit 7.6s, scoring 0.828s per 10k rows
  ocsvm__F_main__s46: fit 6.4s, scoring 0.839s per 10k rows
  ocsvm__F_paper__s42: fit 3.2s, scoring 0.264s per 10k rows
  ocsvm__F_no_user__s42: fit 7.3s, scoring 0.802s per 10k rows
  ocsvm__F_raw__s42: fit 2.1s, scoring 0.376s per 10k rows


In [7]:
# ===== P5 CELL 7: Autoencoder (5 seeds on F_main + 3 ablation feature sets) =====
import copy
from sklearn.neural_network import MLPRegressor
p = MODELS["autoencoder"]

def fit_autoencoder(fs, seed):
    Xtr = load_X(fs, "train")
    scaler = StandardScaler().fit(Xtr)
    Xs = scaler.transform(Xtr).astype(np.float32)
    es_mask = np.zeros(len(Xs), dtype=bool); es_mask[es_idx] = True
    X_fit, X_es = Xs[~es_mask], Xs[es_mask]
    ae = MLPRegressor(hidden_layer_sizes=tuple(p["hidden"]), activation="relu", solver="adam",
                      learning_rate_init=p["lr"], batch_size=p["batch_size"], random_state=seed)
    rng = np.random.default_rng(seed)
    best_loss, best_model, waited, history = np.inf, None, 0, []
    for epoch in range(1, p["max_epochs"] + 1):
        order = rng.permutation(len(X_fit))
        ae.partial_fit(X_fit[order], X_fit[order])
        es_loss = float(np.mean((ae.predict(X_es) - X_es) ** 2))
        history.append(round(es_loss, 6))
        if es_loss < best_loss - 1e-6:
            best_loss, best_model, waited = es_loss, copy.deepcopy(ae), 0
        else:
            waited += 1
            if waited >= p["early_stopping_patience"]:
                break
    print(f"    epochs run: {epoch}, best early-stopping loss: {best_loss:.5f}")
    def score(split):
        X = scaler.transform(load_X(fs, split)).astype(np.float32)
        return np.concatenate([np.mean((best_model.predict(X[i:i + 50000]) - X[i:i + 50000]) ** 2, axis=1)
                               for i in range(0, len(X), 50000)])
    return {"scaler": scaler, "model": best_model, "loss_history": history}, score

print("Autoencoder:")
for fs, seed in configs_for("autoencoder"):
    run_config("autoencoder", fs, seed, fit_autoencoder)

Autoencoder:
    epochs run: 12, best early-stopping loss: 0.13144
  autoencoder__F_main__s42: fit 107.3s, scoring 0.031s per 10k rows
    epochs run: 25, best early-stopping loss: 0.13336
  autoencoder__F_main__s43: fit 325.8s, scoring 0.039s per 10k rows
    epochs run: 19, best early-stopping loss: 0.12284
  autoencoder__F_main__s44: fit 179.7s, scoring 0.035s per 10k rows
    epochs run: 21, best early-stopping loss: 0.10833
  autoencoder__F_main__s45: fit 209.3s, scoring 0.061s per 10k rows
    epochs run: 18, best early-stopping loss: 0.11881
  autoencoder__F_main__s46: fit 152.4s, scoring 0.034s per 10k rows
    epochs run: 14, best early-stopping loss: 0.00002
  autoencoder__F_paper__s42: fit 90.9s, scoring 0.014s per 10k rows
    epochs run: 19, best early-stopping loss: 0.09504
  autoencoder__F_no_user__s42: fit 175.0s, scoring 0.07s per 10k rows
    epochs run: 15, best early-stopping loss: 0.00024
  autoencoder__F_raw__s42: fit 101.7s, scoring 0.015s per 10k rows


In [8]:
# ===== P5 CELL 8: Summary of all saved score files =====
runs = pd.read_json(os.path.join(LOG_DIR, "P5_run_log.jsonl"), lines=True).drop_duplicates("config", keep="last")
files = sorted(f[:-4] for f in os.listdir(SCORE_DIR) if f.endswith(".npz"))
print(f"Score files saved: {len(files)} (expected 27)")
pd.set_option("display.width", 200)
print(runs[["config", "fit_seconds", "score_seconds_per_10k", "peak_ram_gb_so_far"]].to_string(index=False))
runs.to_csv(os.path.join(BASE, "05_results", "P5_compute_cost.csv"), index=False)
slow = runs[runs["score_seconds_per_10k"] >= 10]
print("\nNFR-03 (score 10,000 records in under 10 s):",
      "met by every configuration" if slow.empty else f"NOT met by {', '.join(slow['config'])}")

Score files saved: 27 (expected 27)
                     config  fit_seconds  score_seconds_per_10k  peak_ram_gb_so_far
          random__none__s42          0.0                  0.000                0.51
      rule_userId__none__s0          0.0                  0.000                0.51
           rarity__none__s0          0.2                  0.002                0.51
       iforest__F_main__s42          5.7                  0.290                0.71
       iforest__F_main__s43          3.4                  0.107                0.72
       iforest__F_main__s44          2.7                  0.107                0.72
       iforest__F_main__s45          2.1                  0.105                0.72
       iforest__F_main__s46          2.4                  0.111                0.72
      iforest__F_paper__s42          2.2                  0.062                0.72
    iforest__F_no_user__s42          5.2                  0.090                0.72
        iforest__F_raw__s42          2.5